# 개인실습 — 조회 패턴 변경과 캐시 (7교시) · 정답 모음

> 이 노트북은 `NoSQL_Redis_3_개인실습.ipynb`(문제 노트북)의 **정답**입니다. `NoSQL_Redis_2_핵심개념_데모.ipynb`를 먼저 전부 실행해 둔 상태를 전제로 합니다.


In [ ]:
# ✅ 포인트: 지난 실습과 같은 run_sql()/run_redis() subprocess 헬퍼 + 오늘 새로 쓰는 mongomock
# (이 셀 하나로 노트북이 완결됩니다 — 별도 파일이 필요 없습니다)
# subprocess: 터미널 명령(docker, psql, redis-cli)을 파이썬 안에서 대신 실행해주는 표준 라이브러리
# json/time: 이 셀 자체에는 안 쓰이지만 아래 다른 셀들이 공용으로 쓰므로 여기서 한 번에 import
# io: 문자열(텍스트)을 "파일처럼" 다루게 해주는 표준 라이브러리 — 아래 pd.read_csv(io.StringIO(...))에서 사용
# os: 환경변수(비밀번호 등)를 읽고 설정하는 표준 라이브러리
import subprocess, json, time, io, os
import pandas as pd   # 표 형태(행·열) 데이터를 다루는 대표적인 외부 라이브러리 — SQL 결과를 표로 받을 때 사용
import mongomock       # "진짜 MongoDB" 대신 파이썬 메모리 안에서 흉내 내주는 외부 라이브러리

# 💡 이 노트북을 NoSQL_Redis_1_환경확인.ipynb 없이 단독으로 열어도 동작하도록, db-pg 시작 + db-redis
#    생성(없을 때만)·시작을 여기서 한 번 더 보장합니다. 이미 떠 있으면 순식간에 끝납니다(재실행 안전).
if subprocess.run(["docker", "inspect", "db-redis"], capture_output=True, text=True).returncode != 0:
    subprocess.run(["docker", "run", "-d", "--name", "db-redis", "-p", "6379:6379", "redis:8"],
                    capture_output=True, text=True)
subprocess.run(["docker", "start", "db-pg", "db-redis"], capture_output=True, text=True)

# 함수 안에 """ ... """로 감싼 첫 문장은 "docstring"이라고 부르며, 이 함수가 무슨 일을 하는지 설명하는
# 공식 주석입니다 — help(run_sql)을 실행하면 이 내용이 그대로 출력됩니다.
def run_sql(sql, as_df=True):
    """psql subprocess 헬퍼 — db-pg 컨테이너 안의 psql로 실제 PostgreSQL(course_db)에 실행합니다."""
    # as_df=True 처럼 매개변수에 기본값을 미리 지정해두면, 호출할 때 그 인자를 생략해도 됩니다
    # (예: run_sql("SELECT 1;") 라고만 써도 as_df=True가 자동 적용됩니다).
    # 💡 psql이 이 컴퓨터(호스트)가 아니라 db-pg 컨테이너 "안"에 있으므로, docker exec로 그 안에 들어가
    #    실행합니다 — 이 컴퓨터에 psql을 따로 설치할 필요가 없습니다. -e PGPASSWORD=...는 컨테이너 실행에
    #    비밀번호를 넣어주는 옵션입니다(psql이 비밀번호를 묻지 않고 자동으로 읽어갑니다).
    if as_df:
        # -A(정렬 없이) -F ","(구분자를 콤마로) 옵션으로 CSV 형태 텍스트를 뽑아내는 psql 명령 조합입니다.
        cmd = ["docker", "exec", "-e", "PGPASSWORD=postgres", "db-pg", "psql", "-U", "postgres", "-d", "course_db",
               "-v", "ON_ERROR_STOP=1", "-P", "footer=off", "-A", "-F", ",", "-c", sql]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:  # 0이 아니면 psql 실행이 실패했다는 뜻 — 에러 메시지를 그대로 올려서 알림
            raise RuntimeError(result.stderr)
        text = result.stdout.strip("\n")  # 맨 앞뒤의 줄바꿈만 제거(가운데 내용은 그대로 둠)
        # io.StringIO(text)는 문자열 text를 "디스크에 저장된 CSV 파일"인 것처럼 감싸줍니다.
        # pd.read_csv()는 원래 파일 경로를 받는 함수인데, 이렇게 하면 파일을 실제로 만들지 않고도
        # SQL 결과 텍스트를 곧바로 표(DataFrame)로 변환할 수 있습니다 — text가 비어있으면 빈 표를 반환.
        return pd.read_csv(io.StringIO(text)) if text else pd.DataFrame()
    else:
        cmd = ["docker", "exec", "-e", "PGPASSWORD=postgres", "db-pg", "psql", "-U", "postgres", "-d", "course_db",
               "-v", "ON_ERROR_STOP=1", "-c", sql]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(result.stderr)
        return result.stdout.strip()

# *args 는 "개수가 정해지지 않은 여러 개의 인자를 하나로 묶어서 받는다"는 뜻입니다.
def run_redis(*args):
    """redis-cli subprocess 헬퍼 — db-redis 컨테이너 안의 redis-cli로 실제 Redis에 실행합니다."""
    # [str(a) for a in args]는 args의 각 값을 문자열로 바꾼 새 리스트를 만드는 리스트 컴프리헨션입니다
    # (redis-cli 인자는 모두 문자열이어야 하므로, 숫자를 넣어도 안전하게 문자열로 변환합니다).
    # 💡 redis-cli도 이 컴퓨터가 아니라 db-redis 컨테이너 안에 있으므로 docker exec로 실행합니다.
    cmd = ["docker", "exec", "db-redis", "redis-cli"] + [str(a) for a in args]
    result = subprocess.run(cmd, capture_output=True, text=True)
    out = result.stdout.strip()
    # Redis는 "값이 없음"을 "(nil)"이라는 특수 문자열로 표시합니다 — 이걸 파이썬의 진짜 None으로 바꿔줍니다.
    return None if out == "(nil)" else out

# mongomock.MongoClient()는 "MongoDB 서버에 접속하는 창구"를 만드는 것입니다(실제로는 메모리 안에서 흉내).
mongo_client = mongomock.MongoClient()
mongo_db = mongo_client["course_db"]  # 이 창구 안에서 "course_db"라는 이름의 데이터베이스를 선택

# run_sql("SELECT 1;", as_df=False)는 접속만 확인하는 용도라 결과 텍스트 내용 자체는 중요하지 않고,
# "에러 없이 빈 문자열이 아닌 뭔가가 돌아왔는가"만 봅니다. == ""는 "결과가 빈 문자열과 같은가"를 판정하고,
# 맨 앞의 not이 그 판정을 뒤집습니다 — 즉 "결과가 비어있지 않다"(=연결이 정상이다)를 최종적으로 확인합니다.
print("PostgreSQL 연결 확인:", not run_sql("SELECT 1;", as_df=False) == "")
print("Redis 연결 확인:", run_redis("ping") == "PONG")  # ping 응답이 정확히 "PONG"인지 비교 -> True/False
print("mongomock 준비 완료:", mongo_db.name)

## 도전 1 — 조회 패턴 변경: "태그로 문서 검색"

### 도전 1-A: PostgreSQL에서 태그 검색

**문제 1 정답**

In [ ]:
# ✅ Step 1: tags 컬럼 추가 (TEXT[] 배열)
# ALTER TABLE은 이미 만들어진 테이블의 구조를 바꾸는 SQL 명령입니다(새로 만드는 CREATE TABLE과 다름).
# TEXT[]는 PostgreSQL의 "배열 타입" — 한 셀(칸)에 문자열을 여러 개 담을 수 있습니다(파이썬 리스트와 비슷한 개념).
# IF NOT EXISTS를 붙이면, 이미 이 컬럼이 있을 때 에러를 내지 않고 조용히 넘어갑니다(실습 재실행 대비).
# DEFAULT '{}'는 값을 안 준 기존 행들에 대해 "빈 배열"을 기본값으로 채웁니다(PostgreSQL 배열 리터럴 표기법).
run_sql("""
    ALTER TABLE products
    ADD COLUMN IF NOT EXISTS tags TEXT[] DEFAULT '{}'
""", as_df=False)

# Step 2: 샘플 태그 삽입
# ARRAY['a','b']는 PostgreSQL에서 배열 리터럴(값)을 직접 만드는 문법입니다.
# UPDATE ... SET ... WHERE id = 501은 "id가 501인 행만 골라서 tags 컬럼 값을 바꾼다"는 뜻입니다.
run_sql("""
    UPDATE products SET tags = ARRAY['전자기기','베스트셀러'] WHERE id = 501
""", as_df=False)

# Step 3: 태그 검색
# ANY(tags)는 "배열 tags 안의 원소 중 하나라도"라는 뜻이고, '전자기기' = ANY(tags)는
# "tags 배열 안에 '전자기기'라는 값이 들어있는가"를 묻는 조건입니다(파이썬의 `'전자기기' in tags_list`와 같은 의미).
print(run_sql("""
    SELECT id, name FROM products
    WHERE '전자기기' = ANY(tags)
"""))


**문제 2 정답**

In [ ]:
# ✅ GIN 인덱스 추가 + EXPLAIN으로 실제 실행 계획 확인
# GIN(Generalized Inverted iNdex)은 배열·JSONB처럼 "하나의 칸에 여러 값이 들어있는" 컬럼을 빠르게
# 검색하기 위한 PostgreSQL 전용 인덱스 종류입니다 — 일반 인덱스(B-tree)로는 배열 안의 원소를 빠르게 못 찾습니다.
run_sql("CREATE INDEX IF NOT EXISTS idx_products_tags ON products USING GIN(tags);", as_df=False)
# EXPLAIN을 SELECT 앞에 붙이면 쿼리를 실제로 실행하지 않고, "PostgreSQL이 이 쿼리를 어떤 방법으로 처리할
# 계획인지"(실행 계획)만 보여줍니다 — 인덱스를 실제로 타는지(Index Scan) 그냥 전체를 훑는지(Seq Scan) 확인 가능합니다.
print(run_sql("EXPLAIN SELECT id, name FROM products WHERE '전자기기' = ANY(tags);"))
# 💡 3행짜리 작은 테이블이라 GIN 인덱스가 있어도 플래너가 Seq Scan을 선택합니다.
#    9/22 pgvector에서 본 "작은 테이블은 인덱스를 안 탄다"는 교훈이 여기서도 그대로 재현됩니다.


### 도전 1-B: MongoDB(mongomock)에서 태그 검색

**문제 3 정답**

In [ ]:
# ✅ mongomock — 실제 pymongo와 동일하게 배열 필드를 '포함 여부'로 자동 해석
products_doc = mongo_db["products_doc"]
# "_id": "P501"처럼 _id 값을 직접 지정하면, MongoDB가 자동 생성하는 대신 이 값을 그대로 고유 식별자로 씁니다.
products_doc.insert_one({"_id": "P501", "name": "무선 이어폰", "tags": ["전자기기", "베스트셀러"]})

# -> str, -> list 처럼 함수 정의 끝에 화살표로 타입을 적는 것은 "타입 힌트"입니다 — 실행에는 영향이 없고,
# "이 함수는 이런 타입의 값을 반환할 예정"이라는 사람이 읽기 위한 표시입니다(tag: str도 매개변수 타입 힌트).
def mongo_docs_by_tag(tag: str) -> list:
    """mongomock: 태그 배열에 특정 값이 있는 문서 조회"""
    # find({"tags": tag})처럼 배열 필드에 단일 값을 조건으로 주면, MongoDB는 자동으로
    # "그 배열 안에 이 값이 포함된 문서"를 찾아줍니다(PostgreSQL의 ANY(tags)와 같은 효과를 다른 문법으로 표현).
    return list(products_doc.find({"tags": tag}))

print("mongomock 태그 검색 ->", mongo_docs_by_tag("전자기기"))


(참고: `naive_match` 흔한 실수 설명은 문제 노트북에 이미 포함되어 있어 여기서는 생략합니다.)

### 도전 1-C: Redis에서 태그 검색

**문제 4 정답**

In [ ]:
# ✅ Redis: 태그별 상품 집합을 Set으로 관리
# PostgreSQL의 배열, MongoDB의 배열 필드와 달리, Redis에서는 "태그 하나당 Set 하나"를 만들어
# (예: tag:전자기기:products 라는 키에 그 태그를 가진 상품 ID들을 저장) 태그별 조회를 구현합니다.
def redis_setup_tag_index():
    run_redis("sadd", "tag:전자기기:products", "product:501", "product:502")
    run_redis("sadd", "tag:베스트셀러:products", "product:501")  # product:501은 두 태그 Set에 모두 들어감(중복 가능)

def redis_docs_by_tag(tag: str) -> list:
    key = f"tag:{tag}:products"
    # smembers key: 그 Set에 들어있는 모든 값을 반환합니다(순서는 보장되지 않음 — Set은 원래 순서가 없는 자료구조).
    result = run_redis("smembers", key)
    # result가 None(값이 없음)이거나 빈 문자열이면 빈 리스트를, 아니면 줄 단위로 잘라 리스트로 반환
    return result.splitlines() if result else []

run_redis("del", "tag:전자기기:products", "tag:베스트셀러:products")  # 이전 실습 흔적 삭제 후 새로 시작
redis_setup_tag_index()
print("전자기기 태그 상품 ->", redis_docs_by_tag("전자기기"))


## 도전 2 · Redis 캐시 레이어 추가

**문제 5 정답**

In [ ]:
# pg_recent_views_raw: 캐시를 거치지 않고 매번 PostgreSQL에 직접 물어보는 "느린 버전"
def pg_recent_views_raw(user_id: int, limit: int = 5):
    df = run_sql(f"""
        SELECT user_id, product_id, viewed_at
        FROM view_history WHERE user_id = {user_id}
        ORDER BY viewed_at DESC LIMIT {limit};
    """)
    # to_dict(orient="records")는 pandas 표(DataFrame)를 "행 하나 = 딕셔너리 하나"인 리스트로 바꿔줍니다.
    # 예: [{"user_id":1001, "product_id":"product:501", ...}, {...}, ...] 형태 — JSON으로 저장하기 편한 구조입니다.
    return df.to_dict(orient="records")

# get_recent_views_cached: 4교시에서 배운 Cache-Aside 패턴을 그대로 적용한 "캐시 우선" 버전
def get_recent_views_cached(user_id: int, limit: int = 5, ttl: int = 60) -> dict:
    cache_key = f"recent_views:{user_id}:{limit}"
    cached = run_redis("get", cache_key)
    if cached:
        # 캐시에 있으면 DB를 아예 안 거치고 바로 반환 — json.loads()로 저장된 JSON 문자열을 다시 파이썬 값으로 복원
        return {"source": "cache", "data": json.loads(cached)}
    data = pg_recent_views_raw(user_id, limit)  # 캐시에 없을 때만 위의 "느린 버전" 함수를 호출
    # default=str: json.dumps()가 기본적으로 변환 못 하는 타입(예: 날짜/시각 값)을 만나면 문자열로 바꿔서 저장
    run_redis("set", cache_key, json.dumps(data, ensure_ascii=False, default=str), "EX", str(ttl))
    return {"source": "db", "data": data}

def measure_cache_effect(user_id: int = 1001):
    cache_key = f"recent_views:{user_id}:5"
    run_redis("del", cache_key)  # 캐시를 비운 상태에서 시작 -> 첫 조회는 반드시 db를 타게 됨
    for i in range(3):  # 같은 요청을 3번 반복해서, 캐시가 쌓이면서 속도가 어떻게 달라지는지 관찰
        t0 = time.perf_counter()
        result = get_recent_views_cached(user_id)
        elapsed = (time.perf_counter() - t0) * 1000  # 초 -> 밀리초 단위 변환
        # f"{elapsed:6.1f}"는 전체 폭 6칸을 확보하고 소수점 첫째 자리까지 표시(숫자가 짧으면 앞에 공백으로 채움)
        print(f"조회 {i+1}: {elapsed:6.1f}ms  [source: {result['source']}]")

measure_cache_effect()


**문제 6 정답**

In [ ]:
# benchmark: "캐시 없이 매번 DB만 사용" vs "캐시를 사용" 두 방식의 속도 차이를 직접 재서 비교합니다.
def benchmark(n: int = 10, user_id: int = 1001):
    cache_key = f"recent_views:{user_id}:5"

    # 1) 캐시 없음 시나리오: 매 반복마다 캐시를 강제로 지운 뒤(del) 느린 함수(pg_recent_views_raw)만 호출
    t0 = time.perf_counter()
    for _ in range(n):
        run_redis("del", cache_key)
        pg_recent_views_raw(user_id)
    t_no_cache = (time.perf_counter() - t0) * 1000

    # 2) 캐시 있음 시나리오: 딱 한 번만 비우고, 나머지는 캐시가 쌓인 상태로 반복 호출
    run_redis("del", cache_key)
    t0 = time.perf_counter()
    for _ in range(n):
        get_recent_views_cached(user_id)
    t_with_cache = (time.perf_counter() - t0) * 1000

    print(f"캐시 없음 {n}회: {t_no_cache:.1f}ms")
    print(f"캐시 있음 {n}회: {t_with_cache:.1f}ms")
    # (1 - t_with_cache/t_no_cache)*100은 "캐시가 전체 시간을 몇 % 줄여줬는지" 계산하는 공식입니다
    # — 예를 들어 캐시 있음이 캐시 없음의 20%밖에 안 걸렸다면, 80% 절감된 것입니다.
    print(f"절감: {t_no_cache - t_with_cache:.1f}ms ({(1 - t_with_cache/t_no_cache)*100:.0f}%)")

benchmark()
# 💡 절대값(ms)이 커 보이는 이유: run_sql/run_redis가 매번 psql·redis-cli 프로세스를 새로 띄우기 때문입니다.
#    실무에서는 커넥션을 재사용하는 드라이버(psycopg, redis-py)를 쓰므로 절대값은 훨씬 작습니다.
#    중요한 것은 절대값이 아니라 "캐시가 있으면 훨씬 빠르다"는 방향과 비율입니다.


## 도전 3 · 캐시 무효화

**문제 7 정답**

In [ ]:
# record_view: 새 조회 기록을 남기면서, 이제는 낡은 정보가 된 캐시를 함께 지우는(=무효화하는) 함수
def record_view(user_id: int, product_id: str):
    """새 조회를 PostgreSQL에 기록하고 캐시를 무효화"""
    run_sql(f"""
        INSERT INTO view_history (user_id, product_id) VALUES ({user_id}, '{product_id}')
    """, as_df=False)
    # 캐시 무효화(invalidation): 데이터가 바뀌었으니 기존 캐시 값은 더 이상 맞지 않습니다.
    # 여기서는 새 값을 캐시에 직접 채워 넣지 않고, 그냥 지워버립니다(del) — 그러면 다음 조회 때
    # get_recent_views_cached()가 캐시가 없다고 판단해 자동으로 DB에서 최신 값을 다시 읽어와 캐시를 새로 채웁니다.
    run_redis("del", f"recent_views:{user_id}:5")

before = get_recent_views_cached(1001)
print("기록 전 캐시 소스:", before["source"], "| 개수:", len(before["data"]))

record_view(1001, "product:504")  # 새 조회 기록 -> 내부적으로 캐시가 삭제됨
after = get_recent_views_cached(1001)
# 캐시가 무효화됐으므로 이 호출은 반드시 "db"를 거쳐야 하고(캐시가 비어있으니), 그 결과 캐시가 다시 채워집니다.
print("새 조회 기록 후 캐시 소스:", after["source"], "(무효화됐으므로 db여야 함)")
# [d["product_id"] for d in after["data"]]는 딕셔너리 리스트에서 product_id 값만 뽑아 새 리스트로 만드는
# 리스트 컴프리헨션입니다 — 방금 기록한 product:504가 최신 목록 맨 앞(가장 최근)에 보여야 정상입니다.
print("최신 목록:", [d["product_id"] for d in after["data"]])


## 정리

7개 문제 모두 데모 노트북(Step 1~4)에서 만든 `products`·`view_history`·`users_embedded`·Redis 키를 그대로 재사용해 완성했습니다. 막혔던 문제는 위 코드와 자신이 쓴 코드를 줄 단위로 비교해 보세요.
